<a href="https://colab.research.google.com/github/irezendegarcia-prog/python_sis_26_3_periodo/blob/Italo_Rezende/aula_10/Italo_Rezende_aula_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Atividade Prática 01 - Aula 10: Exportador de Configurações

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Etapa 1: Estrutura em Memória
cluster_config = {
    "nome_cluster": "cluster-producao-01",
    "versao_so": "Ubuntu Server 24.04 LTS",
    "servidores": [
        {
            "servidor": "srv-web-01",
            "ip": "192.168.10.10",
            "portas_liberadas": [80, 443, 22],
        },
        {
            "servidor": "srv-app-02",
            "ip": "192.168.10.11",
            "portas_liberadas": [8080, 8443, 22],
        },
        {
            "servidor": "srv-db-03",
            "ip": "192.168.10.12",
            "portas_liberadas": [5432, 22],
        },
    ],
}

print("Configuração do cluster carregada em memória:")
print(cluster_config)


In [ ]:
# Etapa 2: Exportação JSON
import json

with open("cluster_config.json", "w", encoding="utf-8") as arquivo_json:
    json.dump(cluster_config, arquivo_json, indent=4, ensure_ascii=False)

print("Arquivo 'cluster_config.json' salvo com sucesso.")


In [ ]:
# Etapa 3: Exportação CSV
import csv

with open("servidores_cluster.csv", "w", newline="", encoding="utf-8") as arquivo_csv:
    escritor = csv.writer(arquivo_csv, delimiter=";")
    escritor.writerow(["Servidor", "IP", "Portas"])

    for srv in cluster_config["servidores"]:
        portas_formatadas = ", ".join(str(porta) for porta in srv["portas_liberadas"])
        escritor.writerow([srv["servidor"], srv["ip"], portas_formatadas])

print("Arquivo 'servidores_cluster.csv' salvo com sucesso.")


## Atividade Prática 02 - Aula 10: Inventário de Servidores

In [ ]:
# Etapa 1: Instalação e importação seguindo a PEP 8
!pip install -q openpyxl

import openpyxl
from openpyxl.styles import Font, PatternFill

print(f"openpyxl versão {openpyxl.__version__} importado com sucesso.")


In [ ]:
# Etapa 2: Criação da planilha e cabeçalho
wb_inventario = openpyxl.Workbook()
ws_servidores = wb_inventario.active
ws_servidores.title = "Servidores"

cabecalho = ["Hostname", "IP", "Memória (GB)", "Uso de Disco (%)", "Status"]
ws_servidores.append(cabecalho)

# Destaque visual no cabeçalho
for celula in ws_servidores[1]:
    celula.font = Font(bold=True)

print("Planilha criada com a aba 'Servidores' e cabeçalho adicionado.")


In [ ]:
# Etapa 3: Inserção de dados simulados (mínimo de 4 servidores)
servidores_inventario = [
    ["srv-web-01", "192.168.10.10", 16, 45, "Online"],
    ["srv-app-02", "192.168.10.11", 32, 82, "Alerta"],
    ["srv-db-03", "192.168.10.12", 64, 91, "Crítico"],
    ["srv-bkp-04", "192.168.10.13", 32, 60, "Online"],
]

for servidor in servidores_inventario:
    ws_servidores.append(servidor)

print(f"{len(servidores_inventario)} servidores inseridos na planilha.")


In [ ]:
# Etapa 4: Avisos de uso de disco >= 80% e salvamento
preenchimento_alerta = PatternFill(
    start_color="FFC7CE",
    end_color="FFC7CE",
    fill_type="solid",
)
fonte_alerta = Font(color="9C0006", bold=True)

for linha in ws_servidores.iter_rows(min_row=2, max_row=ws_servidores.max_row):
    hostname = linha[0].value
    celula_disco = linha[3]
    uso_disco = celula_disco.value

    if uso_disco >= 80:
        celula_disco.fill = preenchimento_alerta
        celula_disco.font = fonte_alerta
        print(
            f"[ALERTA] Servidor '{hostname}' com {uso_disco}% de uso de disco: "
            "precisa de expansão de armazenamento."
        )

arquivo_inventario = "inventario_servidores.xlsx"
wb_inventario.save(arquivo_inventario)
print(f"Planilha salva com sucesso em '{arquivo_inventario}'.")


## Atividade Prática 03 - Aula 10: Conversor de Log para Planilha

In [ ]:
# Etapa 1: Abertura do arquivo log.txt com with e tratamento de FileNotFoundError
nome_log = "log.txt"
linhas_log = []

try:
    with open(nome_log, "r", encoding="utf-8") as arquivo_log:
        linhas_log = [linha.strip() for linha in arquivo_log if linha.strip()]
    print(f"Arquivo '{nome_log}' lido com sucesso ({len(linhas_log)} linhas encontradas).")
except FileNotFoundError:
    print(
        f"[AVISO] O arquivo '{nome_log}' não foi encontrado no diretório atual. "
        "Faça o upload do arquivo 'log.txt' (da Aula 09) no painel de Arquivos do Colab e execute novamente."
    )


[AVISO] O arquivo 'log.txt' não foi encontrado no diretório atual. Faça o upload do arquivo 'log.txt' (da Aula 09) no painel de Arquivos do Colab e execute novamente.


In [ ]:
# Etapa 2: Extração de dados (Data/Hora, Tipo de Evento e Mensagem)
registros_log = []

for linha in linhas_log:
    partes = linha.split(" - ", 2)
    if len(partes) == 3:
        data_hora, tipo_evento, mensagem = [parte.strip() for parte in partes]
        registros_log.append((data_hora, tipo_evento, mensagem))
        print(f"Nível extraído: {tipo_evento} | {data_hora}")

print(f"\nTotal de registros extraídos: {len(registros_log)}")



Total de registros extraídos: 0


In [ ]:
# Etapa 3: Criação da planilha auditoria_logs.xlsx com cabeçalho
import openpyxl
from openpyxl.styles import Font

wb_auditoria = openpyxl.Workbook()
ws_auditoria = wb_auditoria.active
ws_auditoria.title = "Auditoria de Logs"

ws_auditoria.append(["Data/Hora", "Tipo de Evento", "Mensagem"])
for celula in ws_auditoria[1]:
    celula.font = Font(bold=True)

print("Planilha de auditoria criada com as colunas: Data/Hora, Tipo de Evento e Mensagem.")


Planilha de auditoria criada com as colunas: Data/Hora, Tipo de Evento e Mensagem.


In [ ]:
# Etapa 4: Preenchimento e salvamento da planilha no diretório de execução
for registro in registros_log:
    ws_auditoria.append(list(registro))

arquivo_saida = "auditoria_logs.xlsx"
wb_auditoria.save(arquivo_saida)
print(f"Planilha '{arquivo_saida}' preenchida e salva com {len(registros_log)} registros.")


Planilha 'auditoria_logs.xlsx' preenchida e salva com 0 registros.
